# Set 05 – scikit-learn-Klassifikation

In diesem Notebook trainieren wir ein erstes Klassifikationsmodell. Wir verwenden den eingebauten Iris-Datensatz, daher ist kein Download nötig.

## Lernziele

- Merkmale (`X`) und Zielvariable (`y`) unterscheiden,
- Trainings- und Testdaten korrekt trennen,
- Vorverarbeitung und Modell in einer Pipeline verbinden,
- ein Modell mit `fit` trainieren und mit `predict` anwenden,
- Klassifikationsmetriken und Kreuzvalidierung einordnen.

## 1. Das Grundprinzip des überwachten Lernens

Beim **überwachten Lernen** kennt das Modell für Trainingsbeispiele die richtige Antwort. Es lernt eine Abbildung von Merkmalen auf ein Ziel.

- `X`: Merkmalsmatrix, Form `(Anzahl Beobachtungen, Anzahl Merkmale)`
- `y`: Zielvariable, ein Wert pro Beobachtung
- **Klassifikation** sagt Kategorien voraus.
- **Regression** sagt numerische Werte voraus.

Die Iris-Aufgabe ist eine Klassifikation: Aus vier Blütenmaßen soll die Pflanzenart vorhergesagt werden.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import sklearn

from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
)
from sklearn.model_selection import cross_validate, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

print("scikit-learn-Version:", sklearn.__version__)

## 2. Daten laden und verstehen

In [ ]:
iris = load_iris(as_frame=True)
X = iris.data
y = iris.target

print("X-Form:", X.shape)
print("y-Form:", y.shape)
print("Klassen:", dict(enumerate(iris.target_names)))
display(X.head())
display(y.value_counts().sort_index().rename(index=dict(enumerate(iris.target_names))))

Vor dem Modellieren prüfen wir Form, Datentypen, fehlende Werte und Klassenverteilung. Der Iris-Datensatz ist klein und ausgeglichen – gut zum Lernen, aber nicht repräsentativ für alle realen ML-Probleme.

In [ ]:
print("Fehlende Werte pro Spalte:\n", X.isna().sum())
display(X.describe())

## 3. Trainings- und Testdaten trennen

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print("Training:", X_train.shape, y_train.shape)
print("Test:    ", X_test.shape, y_test.shape)

Das Modell darf die Testdaten während des Trainings nicht sehen. Sonst messen wir eher Erinnerungsvermögen als Generalisierung. `stratify=y` erhält ungefähr die Klassenanteile, `random_state` macht die Aufteilung reproduzierbar.

## 4. Pipeline erstellen und Modell trainieren

In [ ]:
modell = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000),
)

modell.fit(X_train, y_train)
print(modell)

Der `StandardScaler` zentriert jedes Merkmal und skaliert es auf eine vergleichbare Größenordnung. Die Pipeline lernt die Skalierung **nur aus den Trainingsdaten** und wendet sie konsistent auf neue Daten an. Dadurch vermeiden wir Data Leakage.

Die einheitliche scikit-learn-Schnittstelle ist zentral:

- `fit(X, y)`: Parameter aus Trainingsdaten lernen
- `predict(X)`: Klassen vorhersagen
- `predict_proba(X)`: Klassenwahrscheinlichkeiten liefern
- `score(X, y)`: Standardmetrik des Modells berechnen

## 5. Auf bisher unbekannten Testdaten bewerten

In [ ]:
y_pred = modell.predict(X_test)
genauigkeit = accuracy_score(y_test, y_pred)

print(f"Accuracy: {genauigkeit:.3f}")
print("\nKlassifikationsbericht:\n")
print(classification_report(y_test, y_pred, target_names=iris.target_names))

**Accuracy** ist der Anteil korrekter Vorhersagen. Bei stark unausgeglichenen Klassen reicht sie oft nicht. `precision` fragt vereinfacht: Wie viele Vorhersagen dieser Klasse waren richtig? `recall`: Wie viele tatsächlich zugehörige Fälle wurden gefunden? Der F1-Wert verbindet beide.

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred,
    display_labels=iris.target_names,
    cmap="Blues",
)
plt.title("Konfusionsmatrix auf den Testdaten")
plt.show()

Die Konfusionsmatrix zeigt, welche Klassen miteinander verwechselt werden. Zeilen stehen standardmäßig für tatsächliche, Spalten für vorhergesagte Klassen.

## 6. Stabilere Schätzung mit Kreuzvalidierung

In [ ]:
cv_ergebnis = cross_validate(
    modell,
    X,
    y,
    cv=5,
    scoring=["accuracy", "f1_macro"],
)

cv_tabelle = pd.DataFrame(cv_ergebnis)[["test_accuracy", "test_f1_macro"]]
display(cv_tabelle)
print("Mittlere CV-Accuracy:", cv_tabelle["test_accuracy"].mean().round(3))
print("Standardabweichung:    ", cv_tabelle["test_accuracy"].std().round(3))

Bei der Kreuzvalidierung wird mehrfach mit unterschiedlichen Trainings- und Validierungsanteilen gelernt. Das Ergebnis ist meist aussagekräftiger als eine einzelne Aufteilung. Ein finaler, unangetasteter Testsatz bleibt bei ernsthaften Projekten trotzdem sinnvoll.

## 7. Eine neue Blüte vorhersagen

In [ ]:
neue_bluete = pd.DataFrame(
    [[5.9, 3.0, 5.1, 1.8]],
    columns=X.columns,
)

klasse = modell.predict(neue_bluete)[0]
wahrscheinlichkeiten = modell.predict_proba(neue_bluete)[0]

print("Vorhergesagte Art:", iris.target_names[klasse])
display(pd.Series(wahrscheinlichkeiten, index=iris.target_names, name="Wahrscheinlichkeit"))

Modellwahrscheinlichkeiten sind keine automatische Garantie für Verlässlichkeit. Sie hängen von Modell, Datenqualität, Repräsentativität und Kalibrierung ab. Vorhersagen außerhalb des bekannten Datenbereichs sind besonders kritisch.

## Übung

1. Ändere `random_state` und beobachte die Test-Accuracy.
2. Trainiere eine Pipeline mit `KNeighborsClassifier(n_neighbors=5)` statt logistischer Regression.
3. Vergleiche beide Modelle mit derselben 5-fachen Kreuzvalidierung.

Wichtig: Vergleiche Modelle mit identischen Splits und derselben Metrik.

In [ ]:
# Musterlösung für Aufgabe 2 und 3
from sklearn.model_selection import cross_val_score
from sklearn.neighbors import KNeighborsClassifier

knn_modell = make_pipeline(
    StandardScaler(),
    KNeighborsClassifier(n_neighbors=5),
)

log_scores = cross_val_score(modell, X, y, cv=5, scoring="accuracy")
knn_scores = cross_val_score(knn_modell, X, y, cv=5, scoring="accuracy")

vergleich = pd.DataFrame({
    "Logistische Regression": log_scores,
    "KNN": knn_scores,
})
display(vergleich)
display(vergleich.agg(["mean", "std"]))

## Typischer ML-Ablauf

1. Problem und Zielmetrik definieren.
2. Daten verstehen und Qualität prüfen.
3. Testdaten abtrennen.
4. Vorverarbeitung und Modell als Pipeline bauen.
5. Mit Trainingsdaten lernen und über Kreuzvalidierung vergleichen.
6. Einmalig auf Testdaten bewerten.
7. Ergebnisse, Grenzen und Annahmen dokumentieren.

Ein hoher Metrikwert allein bedeutet noch kein gutes Produkt: Datenleckage, Verzerrungen, falsche Zielgrößen und veränderte reale Daten können ein Modell unbrauchbar machen.